In [4]:
import os
from pathlib import Path

# 1. Start from the project root
project_root = Path(os.getcwd()).parent
target_file = "RVS_MASTER_SILVER_STANDARD.parquet"

found_path = None
print(f"Searching for {target_file} starting from {project_root}...")

for root, dirs, files in os.walk(project_root):
    if target_file in files:
        found_path = os.path.join(root, target_file)
        print(f"🎯 FOUND IT! Use this path: {found_path}")
        break

if not found_path:
    print("❌ Still not found. Here are the top-level files in root:")
    print(os.listdir(project_root))

Searching for RVS_MASTER_SILVER_STANDARD.parquet starting from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning...
🎯 FOUND IT! Use this path: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\RVS_MASTER_SILVER_STANDARD.parquet


In [11]:
print(silver_df.columns.tolist())

['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun', 'target_tags', 'city_source']


In [17]:
import json
import os

cities = ["manhattan", "pittsburgh", "philadelphia"]
all_gt_data = {}

for city in cities:
    json_path = f"../data/{city}/{city}.json"
    
    if os.path.exists(json_path):
        count = 0
        with open(json_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    # Key by sample number to match silver_df
                    s_id = str(item['rvs_sample_number'])
                    all_gt_data[s_id] = item['rvs_goal_point']
                    count += 1
                except Exception as e:
                    print(f"Skipping bad line in {city}: {e}")
        print(f"✅ Loaded {count} samples from {city}/{city}.json")
    else:
        print(f"⚠️ Warning: File not found at {json_path}")

# Map and check
answerable_df['gold_goal_coords'] = answerable_df['sample_id'].map(all_gt_data)
matched_count = answerable_df['gold_goal_coords'].notna().sum()

print(f"\n--- 📊 Final Sync ---")
print(f"Successfully Matched: {matched_count} / {len(answerable_df)}")

# Filter for the final comparison_df
comparison_df = answerable_df.dropna(subset=['gold_goal_coords']).copy()

✅ Loaded 7000 samples from manhattan/manhattan.json
✅ Loaded 1023 samples from pittsburgh/pittsburgh.json
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad line in philadelphia: 'rvs_sample_number'
Skipping bad l

In [18]:
import json
import os

cities = ["manhattan", "pittsburgh", "philadelphia"]
all_gt_data = {}

for city in cities:
    json_path = f"../data/{city}/{city}.json"
    
    if os.path.exists(json_path):
        count = 0
        skipped = 0
        with open(json_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    item = json.loads(line)
                    # Check if the key exists in this specific JSON line
                    if 'rvs_sample_number' in item and 'rvs_goal_point' in item:
                        s_id = str(item['rvs_sample_number'])
                        all_gt_data[s_id] = item['rvs_goal_point']
                        count += 1
                    else:
                        skipped += 1
                except Exception:
                    skipped += 1
        
        print(f"✅ {city.capitalize()}: Loaded {count} samples (Skipped {skipped} non-data lines)")
    else:
        print(f"⚠️ Warning: File not found at {json_path}")

# Map to your DataFrame
answerable_df['gold_goal_coords'] = answerable_df['sample_id'].map(all_gt_data)
matched_count = answerable_df['gold_goal_coords'].notna().sum()

print(f"\n--- 📊 Final Sync ---")
print(f"Successfully Matched: {matched_count} / {len(answerable_df)}")

# Create comparison_df
comparison_df = answerable_df.dropna(subset=['gold_goal_coords']).copy()

✅ Manhattan: Loaded 7000 samples (Skipped 0 non-data lines)
✅ Pittsburgh: Loaded 1023 samples (Skipped 0 non-data lines)
✅ Philadelphia: Loaded 0 samples (Skipped 1278 non-data lines)

--- 📊 Final Sync ---
Successfully Matched: 6228 / 7263


In [19]:
with open("../data/philadelphia/philadelphia.json", 'r') as f:
    for line in f:
        if line.strip():
            print(json.loads(line))
            break

{'content': "Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street, on the block with a Cinemark cinema. An Acme supermarket is north on the next block. ", 'rvs_goal_point': [39.9534919, -75.2028854], 'key': 9126, 'region': 'Philadelphia', 'rvs_start_point': [39.9546925, -75.1832927]}


In [20]:
import json
import os

cities = ["manhattan", "pittsburgh", "philadelphia"]
all_gt_data = {}

for city in cities:
    json_path = f"../data/{city}/{city}.json"
    if not os.path.exists(json_path):
        continue
        
    count = 0
    with open(json_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            
            # ID Logic: Manhattan/Pitt use 'rvs_sample_number', Philly uses 'key'
            s_id = str(item.get('rvs_sample_number') if item.get('rvs_sample_number') is not None else item.get('key'))
            
            # Goal Logic: All seem to use 'rvs_goal_point' based on your snippet
            goal = item.get('rvs_goal_point')
            
            if s_id and goal:
                all_gt_data[s_id] = goal
                count += 1
                
    print(f"✅ {city.capitalize()}: Loaded {count} samples")

# 1. Map Human Goals to the Silver Standard
answerable_df['gold_goal_coords'] = answerable_df['sample_id'].map(all_gt_data)

# 2. Filter for final comparison
comparison_df = answerable_df.dropna(subset=['gold_goal_coords']).copy()

# 3. Split coordinates for the distance calculation
# Human Goal: [lat, lon]
comparison_df['human_lat'] = comparison_df['gold_goal_coords'].apply(lambda x: x[0])
comparison_df['human_lon'] = comparison_df['gold_goal_coords'].apply(lambda x: x[1])

# Oracle Goal: Assuming you have 'oracle_lat' and 'oracle_lon' from your previous graph lookup
# If not, you'll need to map your 'gold_goal_node' to coordinates here.

print(f"\n--- 📊 Final Sync ---")
print(f"Successfully Matched: {len(comparison_df)} / {len(answerable_df)}")

✅ Manhattan: Loaded 7000 samples
✅ Pittsburgh: Loaded 1023 samples
✅ Philadelphia: Loaded 1278 samples

--- 📊 Final Sync ---
Successfully Matched: 6228 / 7263


In [23]:
import pickle
import os

graph_dict = {}

# Define the exact filenames you have in those subfolders
city_files = {
    "manhattan": "manhattan_graph.gpickle",
    "philadelphia": "philadelphia_graph.gpickle",
    "pittsburgh": "pittsburgh_small_graph.gpickle" 
}

for city, filename in city_files.items():
    path = f"../data/{city}/{filename}"
    
    if os.path.exists(path):
        print(f"Loading {filename}...")
        with open(path, 'rb') as f:
            graph_dict[city] = pickle.load(f)
    else:
        print(f"⚠️ Warning: Could not find {path}")

print(f"✅ Loaded {len(graph_dict)} graphs.")

Loading manhattan_graph.gpickle...


C:\Users\adan\AppData\Local\Temp\ipykernel_4892\1633857166.py:19: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  graph_dict[city] = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_4892\1633857166.py:19: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  graph_dict[city] = pickle.load(f)


Loading philadelphia_graph.gpickle...
Loading pittsburgh_small_graph.gpickle...
✅ Loaded 3 graphs.


In [24]:
def get_node_lat_lon(row, graphs):
    city_key = str(row['city']).lower()
    G = graphs.get(city_key)
    
    if G is None:
        return pd.Series([None, None])
    
    node_id = row['gold_goal_node']
    
    # Try looking up as-is, then try as integer if it's a string
    try:
        if node_id in G.nodes:
            return pd.Series([G.nodes[node_id]['y'], G.nodes[node_id]['x']])
        
        # Try integer conversion if node_id is a string
        int_id = int(float(node_id)) 
        if int_id in G.nodes:
            return pd.Series([G.nodes[int_id]['y'], G.nodes[int_id]['x']])
    except:
        pass
        
    return pd.Series([None, None])

print("Mapping Oracle node IDs to coordinates...")
comparison_df[['oracle_lat', 'oracle_lon']] = comparison_df.apply(
    get_node_lat_lon, axis=1, graphs=graph_dict
)

# Check the match rate
valid_coords = comparison_df['oracle_lat'].notna().sum()
print(f"✅ Successfully mapped {valid_coords} / {len(comparison_df)} Oracle coordinates.")

Mapping Oracle node IDs to coordinates...
✅ Successfully mapped 5451 / 6228 Oracle coordinates.


In [25]:
from src.utils import haversine_vectorized

# 1. Error calculation
comparison_df['dist_error_m'] = haversine_vectorized(
    comparison_df['oracle_lat'], comparison_df['oracle_lon'],
    comparison_df['human_lat'], comparison_df['human_lon']
)

# 2. Convergence threshold
comparison_df['is_convergent'] = comparison_df['dist_error_m'] <= 50

# 3. Final metrics
print(f"\n--- 📈 CONVERGENCE AUDIT ---")
print(f"Convergence Accuracy: {comparison_df['is_convergent'].mean():.2%}")
print(f"Median Error: {comparison_df['dist_error_m'].median():.2f}m")


--- 📈 CONVERGENCE AUDIT ---
Convergence Accuracy: 1.35%
Median Error: 506512.06m


In [26]:
# Create a test column with swapped coordinates
comparison_df['dist_error_swapped'] = haversine_vectorized(
    comparison_df['oracle_lon'], comparison_df['oracle_lat'], # Swapped
    comparison_df['human_lat'], comparison_df['human_lon']
)

print(f"Median Error (Swapped): {comparison_df['dist_error_swapped'].median():.2f}m")
print(f"Convergence if Swapped: {(comparison_df['dist_error_swapped'] <= 50).mean():.2%}")

Median Error (Swapped): 15228120.22m
Convergence if Swapped: 0.00%


In [27]:
# Look at the first 3 rows to spot the drift
debug_cols = ['city', 'oracle_lat', 'oracle_lon', 'human_lat', 'human_lon', 'dist_error_m']
print(comparison_df[debug_cols].head(3))

            city  oracle_lat  oracle_lon  human_lat  human_lon  dist_error_m
1278  pittsburgh         NaN         NaN  40.451332 -79.983414           NaN
1279  pittsburgh         NaN         NaN  40.439167 -79.963611           NaN
1280  pittsburgh         NaN         NaN  40.449256 -79.986622           NaN


In [28]:
G_pitt = graph_dict.get('pittsburgh')
sample_ids_in_df = comparison_df[comparison_df['city'] == 'pittsburgh']['gold_goal_node'].head(5).tolist()

print(f"Sample IDs from your Dataframe: {sample_ids_in_df}")
if G_pitt:
    print(f"Are they in the Pittsburgh graph? {[id in G_pitt.nodes for id in sample_ids_in_df]}")
    print(f"Total nodes in Pitt graph: {len(G_pitt.nodes)}")
    print(f"Example node ID from Graph: {list(G_pitt.nodes)[0]}")

Sample IDs from your Dataframe: ['#6076721414', '#374188455', '#651244961', '#5205450390', '#2135209284']
Are they in the Pittsburgh graph? [False, False, False, False, False]
Total nodes in Pitt graph: 34
Example node ID from Graph: #203557513


In [30]:
import osmnx as ox

print("Downloading full Pittsburgh drive network (this is faster than 777 individual fetches)...")
# 'drive' matches most RVS datasets, but you can use 'all' to be safe
G_pitt_full = ox.graph_from_place("Pittsburgh, Pennsylvania, USA", network_type='drive')

# Add it to your graph dictionary
graph_dict['pittsburgh'] = G_pitt_full

print(f"✅ Loaded full graph with {len(G_pitt_full.nodes)} nodes.")

✅ Loaded full graph with 9106 nodes.


In [32]:
import pandas as pd

def get_node_lat_lon_v3(row, graphs):
    city_key = str(row['city']).lower()
    G = graphs.get(city_key)
    if G is None: 
        return pd.Series([None, None])
    
    node_id = row['gold_goal_node']
    
    # Create a list of potential ID formats to check in the graph
    # 1. As is ('#123')
    # 2. Stripped of # ('123')
    # 3. As an integer (123)
    lookups = [node_id]
    if isinstance(node_id, str):
        clean_id = node_id.replace('#', '')
        lookups.append(clean_id)
        try:
            lookups.append(int(clean_id))
        except:
            pass
            
    for l in lookups:
        if l in G.nodes:
            # OSMnx uses 'y' for Latitude and 'x' for Longitude
            return pd.Series([G.nodes[l]['y'], G.nodes[l]['x']])
            
    return pd.Series([None, None])

print("Mapping Oracle node IDs to coordinates using full graphs...")
comparison_df[['oracle_lat', 'oracle_lon']] = comparison_df.apply(
    get_node_lat_lon_v3, axis=1, graphs=graph_dict
)

# Final cleanup
final_comparison = comparison_df.dropna(subset=['oracle_lat', 'oracle_lon']).copy()
print(f"✅ Successfully matched {len(final_comparison)} Oracle nodes.")

Mapping Oracle node IDs to coordinates using full graphs...
✅ Successfully matched 5450 Oracle nodes.


In [33]:
from src.utils import haversine_vectorized

# Calculate error
final_comparison['dist_error_m'] = haversine_vectorized(
    final_comparison['oracle_lat'], final_comparison['oracle_lon'],
    final_comparison['human_lat'], final_comparison['human_lon']
)

# Convergence check
final_comparison['is_convergent'] = final_comparison['dist_error_m'] <= 50

print(f"\n--- 📈 AUDIT RESULTS ---")
print(f"Total Samples: {len(final_comparison)}")
print(f"Convergence Accuracy (<50m): {final_comparison['is_convergent'].mean():.2%}")
print(f"Median Error: {final_comparison['dist_error_m'].median():.2f} meters")


--- 📈 AUDIT RESULTS ---
Total Samples: 5450
Convergence Accuracy (<50m): 1.54%
Median Error: 506512.31 meters


In [35]:
import json
import os

cities = ["manhattan", "pittsburgh", "philadelphia"]
all_gt_data_unique = {}

for city in cities:
    json_path = f"../data/{city}/{city}.json"
    if not os.path.exists(json_path): continue
    
    with open(json_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            # Standardize the ID based on what we found earlier
            s_id = str(item.get('rvs_sample_number') if item.get('rvs_sample_number') is not None else item.get('key'))
            
            # Create a UNIQUE key like 'pittsburgh_1278'
            unique_key = f"{city.lower()}_{s_id}"
            all_gt_data_unique[unique_key] = item.get('rvs_goal_point')

print(f"✅ Rebuilt ground truth with {len(all_gt_data_unique)} unique city-sample keys.")

✅ Rebuilt ground truth with 2251 unique city-sample keys.


In [37]:
# 1. Ensure oracle_lat/lon are in the main dataframe
# If they are only in comparison_df, we can bring them back via the index
answerable_df['oracle_lat'] = comparison_df['oracle_lat']
answerable_df['oracle_lon'] = comparison_df['oracle_lon']

# 2. Build the unique composite key in your DataFrame
# This prevents Philly IDs from overwriting Pittsburgh IDs
answerable_df['unique_id'] = answerable_df['city'].str.lower() + "_" + answerable_df['sample_id'].astype(str)

# 3. Map the Human Goals using the UNIQUE dictionary we built in the last step
answerable_df['gold_goal_coords'] = answerable_df['unique_id'].map(all_gt_data_unique)

# 4. Now filter: we need rows that have BOTH an Oracle choice AND a Human ground truth
final_comparison = answerable_df.dropna(subset=['gold_goal_coords', 'oracle_lat']).copy()

# 5. Extract Human Lat/Lon for the final math
final_comparison['human_lat'] = final_comparison['gold_goal_coords'].apply(lambda x: x[0])
final_comparison['human_lon'] = final_comparison['gold_goal_coords'].apply(lambda x: x[1])

print(f"✅ Final Audit Set Ready: {len(final_comparison)} valid samples.")

✅ Final Audit Set Ready: 5450 valid samples.


In [38]:
from src.utils import haversine_vectorized

# Calculate distance between Oracle's node and Human's point
final_comparison['dist_error_m'] = haversine_vectorized(
    final_comparison['oracle_lat'], final_comparison['oracle_lon'],
    final_comparison['human_lat'], final_comparison['human_lon']
)

# Convergence check (50 meters)
final_comparison['is_convergent'] = final_comparison['dist_error_m'] <= 50

print(f"--- 📈 FINAL CORRECTED AUDIT ---")
print(f"Convergence Accuracy (<50m): {final_comparison['is_convergent'].mean():.2%}")
print(f"Median Error: {final_comparison['dist_error_m'].median():.2f} meters")

# If you want to see the "Best" matches:
print("\nTop 5 Closest Matches:")
print(final_comparison[['city', 'sample_id', 'dist_error_m']].nsmallest(5, 'dist_error_m'))

--- 📈 FINAL CORRECTED AUDIT ---
Convergence Accuracy (<50m): 8.73%
Median Error: 2453.46 meters

Top 5 Closest Matches:
           city sample_id  dist_error_m
2408  manhattan       406           0.0
2558  manhattan       306           0.0
2626  manhattan       397           0.0
2649  manhattan       200           0.0
2714  manhattan      6292           0.0


In [39]:
# Check if many 'wrong' samples point to the same lat/lon
print("Most frequent Oracle coordinates:")
print(final_comparison[final_comparison['dist_error_m'] > 50][['oracle_lat', 'oracle_lon']].value_counts().head(5))

Most frequent Oracle coordinates:
oracle_lat  oracle_lon
40.750416   -73.967706    6
40.739738   -73.996817    5
40.722845   -73.994228    5
40.734805   -74.006939    4
40.774224   -73.990503    4
Name: count, dtype: int64


In [40]:
city_stats = final_comparison.groupby('city').agg(
    accuracy=('is_convergent', 'mean'),
    median_error_m=('dist_error_m', 'median'),
    sample_count=('sample_id', 'count')
)
print(city_stats)

           accuracy  median_error_m  sample_count
city                                             
manhattan  0.087339     2453.462753          5450


In [41]:
near_miss_acc = (final_comparison['dist_error_m'] <= 500).mean()
print(f"Soft Accuracy (<500m): {near_miss_acc:.2%}")

Soft Accuracy (<500m): 11.67%


In [42]:
import numpy as np

# 1. Get all possible lats/lons from the Manhattan graph
all_nodes = graph_dict['manhattan'].nodes(data=True)
lats = [d['y'] for n, d in all_nodes]
lons = [d['x'] for n, d in all_nodes]

# 2. Assign a random node to every row in your final_comparison
final_comparison['random_lat'] = np.random.choice(lats, size=len(final_comparison))
final_comparison['random_lon'] = np.random.choice(lons, size=len(final_comparison))

# 3. Calculate Random Error
final_comparison['random_error_m'] = haversine_vectorized(
    final_comparison['random_lat'], final_comparison['random_lon'],
    final_comparison['human_lat'], final_comparison['human_lon']
)

print(f"Model Median Error: {final_comparison['dist_error_m'].median():.2f}m")
print(f"Random Baseline Median Error: {final_comparison['random_error_m'].median():.2f}m")

Model Median Error: 2453.46m
Random Baseline Median Error: 2933.92m


In [43]:
# 1. Clear Successes
print("--- ✅ CLEAR SUCCESSES (Model was spot on) ---")
successes = final_comparison[final_comparison['dist_error_m'] < 50].head(3)
for i, row in successes.iterrows():
    print(f"ID: {row['sample_id']} | Dist: {row['dist_error_m']:.2f}m")
    # If 'content' or 'instruction' is in your df, print it here:
    # print(f"Text: {row['content']}\n")

print("\n" + "-"*30 + "\n")

# 2. The "Anxiety" Samples (Near the median)
print("--- ⚠️ MEDIAN ERRORS (General neighborhood only) ---")
medians = final_comparison[(final_comparison['dist_error_m'] > 2000) & (final_comparison['dist_error_m'] < 3000)].head(3)
for i, row in medians.iterrows():
    print(f"ID: {row['sample_id']} | Dist: {row['dist_error_m']:.2f}m")
    # print(f"Text: {row['content']}\n")

--- ✅ CLEAR SUCCESSES (Model was spot on) ---
ID: 406 | Dist: 0.00m
ID: 306 | Dist: 0.00m
ID: 397 | Dist: 0.00m

------------------------------

--- ⚠️ MEDIAN ERRORS (General neighborhood only) ---
ID: 316 | Dist: 2472.28m
ID: 457 | Dist: 2435.06m
ID: 328 | Dist: 2112.45m


In [44]:
# Create a list of the IDs we want to inspect
target_ids = ['316', '457', '328']

# Filter the original dataframe to find the instructions
# (Assuming your instruction column is called 'content' or 'instruction')
inspection = answerable_df[answerable_df['sample_id'].isin(target_ids)]

for i, row in inspection.iterrows():
    print(f"ID: {row['sample_id']}")
    print(f"Instruction: {row.get('content') or row.get('instruction')}")
    print(f"Oracle Lat/Lon: {row['oracle_lat']}, {row['oracle_lon']}")
    print("-" * 30)

ID: 316
Instruction: Meet me at the parking entrance on West General Robinson Street. Find the train station on the same street. It is diagonally across the street from it. It is a parking structure, not a lot. The parking entrance is located on the north west corner of the building.
Oracle Lat/Lon: nan, nan
------------------------------
ID: 457
Instruction: Head east and meet up with me at the cafe on Baum Boulevatrd. It's a block away (almost) and west of a pharmacy. There is a fairly major grocery store on the corner just east of it.
Oracle Lat/Lon: nan, nan
------------------------------
ID: 328
Instruction: Cross the river and come to McMasters Way. I'm in a restaurant on that street near a market square. You will see Starbucks and Burke Building in the block to the south of the restaurant across an avenue.
Oracle Lat/Lon: nan, nan
------------------------------
ID: 457
Instruction: Head up northwest to the restaurant. It's on Graeme St. I'm at the northwest corner top of the mar

In [45]:
# 1. Re-run your loading logic for the Silver Standard
# answerable_df = pd.read_parquet('path_to_your_file.parquet') 

# 2. Re-apply the unique ID and mapping
answerable_df['unique_id'] = answerable_df['city'].str.lower() + "_" + answerable_df['sample_id'].astype(str)
answerable_df['gold_goal_coords'] = answerable_df['unique_id'].map(all_gt_data_unique)

# 3. Re-Verify
print(f"Columns: {answerable_df.columns}")
print(f"Total Rows: {len(answerable_df)}")
print(f"Matched Goals: {answerable_df['gold_goal_coords'].notna().sum()}")

Columns: Index(['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count',
       'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun',
       'target_tags', 'city_source', 'gold_goal_coords', 'unique_id',
       'oracle_lat', 'oracle_lon'],
      dtype='str')
Total Rows: 7263
Matched Goals: 6228


In [46]:
# 1. Filter for rows that have both Oracle and Human coordinates
# This removes the 'NaN' noise we saw in the Pittsburgh samples
final_audit = answerable_df.dropna(subset=['oracle_lat', 'gold_goal_coords']).copy()

# 2. Extract Human Lat/Lon from the goal_coords list
final_audit['human_lat'] = final_audit['gold_goal_coords'].apply(lambda x: x[0])
final_audit['human_lon'] = final_audit['gold_goal_coords'].apply(lambda x: x[1])

# 3. Calculate distance
from src.utils import haversine_vectorized
final_audit['dist_error_m'] = haversine_vectorized(
    final_audit['oracle_lat'], final_audit['oracle_lon'],
    final_audit['human_lat'], final_audit['human_lon']
)

# 4. Success metrics
final_audit['is_convergent'] = final_audit['dist_error_m'] <= 50
final_audit['is_soft_match'] = final_audit['dist_error_m'] <= 500

print(f"--- 🎯 CLEAN AUDIT RESULTS ({len(final_audit)} samples) ---")
print(f"Perfect Convergence (<50m): {final_audit['is_convergent'].mean():.2%}")
print(f"Neighborhood Match (<500m): {final_audit['is_soft_match'].mean():.2%}")
print(f"Median Error: {final_audit['dist_error_m'].median():.2f} meters")

# Break it down by city to see where the model shines
print("\n--- 🏙️ Performance by City ---")
print(final_audit.groupby('city')['dist_error_m'].median())

--- 🎯 CLEAN AUDIT RESULTS (5450 samples) ---
Perfect Convergence (<50m): 8.73%
Neighborhood Match (<500m): 11.67%
Median Error: 2453.46 meters

--- 🏙️ Performance by City ---
city
manhattan    2453.462753
Name: dist_error_m, dtype: float64


In [48]:
# 1. Force identify the dataframe (fixes the Pylance 'Undefined' issue)
# Ensure the variable name matches exactly throughout this cell
df = answerable_df 

# 2. Re-map START coordinates directly into the audit dataframe
print("Mapping Start and Goal coordinates...")

def get_coords(node_id, city, graphs):
    # Re-using the logic from v3
    city_key = str(city).lower()
    G = graphs.get(city_key)
    if G is None or pd.isna(node_id): return pd.Series([None, None])
    
    clean_id = str(node_id).replace('#', '')
    lookups = [node_id, clean_id]
    try: lookups.append(int(clean_id))
    except: pass
        
    for l in lookups:
        if l in G.nodes:
            return pd.Series([G.nodes[l]['y'], G.nodes[l]['x']])
    return pd.Series([None, None])

# Map Start Nodes
df[['start_lat', 'start_lon']] = df.apply(
    lambda r: get_coords(r['start_node'], r['city'], graph_dict), axis=1
)

# Map Oracle Choice Nodes (if not already there)
df[['oracle_lat', 'oracle_lon']] = df.apply(
    lambda r: get_coords(r['gold_goal_node'], r['city'], graph_dict), axis=1
)

# 3. Create the unique ID and Map the Human Ground Truth
df['unique_id'] = df['city'].str.lower() + "_" + df['sample_id'].astype(str)
df['human_goal_coords'] = df['unique_id'].map(all_gt_data_unique)

# 4. Filter for a clean Audit Set
audit = df.dropna(subset=['oracle_lat', 'human_goal_coords', 'start_lat']).copy()

# Extract Human Lats
audit['human_lat'] = audit['human_goal_coords'].apply(lambda x: x[0])
audit['human_lon'] = audit['human_goal_coords'].apply(lambda x: x[1])

print(f"✅ Audit set ready with {len(audit)} samples.")

Mapping Start and Goal coordinates...
✅ Audit set ready with 5450 samples.


In [49]:
from src.utils import haversine_vectorized

# Error relative to where the human wants to go
audit['error_to_goal'] = haversine_vectorized(
    audit['oracle_lat'], audit['oracle_lon'],
    audit['human_lat'], audit['human_lon']
)

# Error relative to where the journey started
audit['error_to_start'] = haversine_vectorized(
    audit['oracle_lat'], audit['oracle_lon'],
    audit['start_lat'], audit['start_lon']
)

print(f"Median Error to Human Goal: {audit['error_to_goal'].median():.2f}m")
print(f"Median Error to Start Point: {audit['error_to_start'].median():.2f}m")

Median Error to Human Goal: 2453.46m
Median Error to Start Point: 1134.95m


In [50]:
print(f"Avg Lat Shift: {(audit['oracle_lat'] - audit['human_lat']).mean():.5f}")

Avg Lat Shift: -0.00020


In [51]:
# Is the model closer to the destination than the beginning?
audit['heading_correct'] = audit['error_to_goal'] < audit['error_to_start']

print(f"Percentage of samples where Oracle is closer to Goal than Start: {audit['heading_correct'].mean():.2%}")

# Look at the 'Success' cases vs 'Fail' cases
print("\nMedian error when heading is correct:")
print(f"{audit[audit['heading_correct']]['error_to_goal'].median():.2f}m")

Percentage of samples where Oracle is closer to Goal than Start: 22.33%

Median error when heading is correct:
448.54m


In [52]:
# Calculate distance between the Start Node and the Human Goal
audit['human_trip_dist'] = haversine_vectorized(
    audit['start_lat'], audit['start_lon'],
    audit['human_lat'], audit['human_lon']
)

print(f"Median Human Trip Distance: {audit['human_trip_dist'].median():.2f}m")

Median Human Trip Distance: 2419.29m


The "Golden Subset" Validation (after notebook reset)

In [9]:
import pandas as pd
import numpy as np
import os

# 1. Self-contained Haversine
def simple_haversine(lat1, lon1, lat2, lon2):
    R = 6371000 
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# 2. Load File
path = '../data/RVS_MASTER_SILVER_STANDARD.parquet'
if os.path.exists(path):
    df = pd.read_parquet(path)
    print("--- 📂 FILE STRUCTURE ---")
    print(f"Columns found: {df.columns.tolist()}")
    
    # 3. Dynamic Column Mapping
    # We look for whatever names you used for Lat/Lon
    lat_cols = [c for c in df.columns if 'lat' in c.lower()]
    lon_cols = [c for c in df.columns if 'lon' in c.lower()]
    print(f"Detected Lat columns: {lat_cols}")
    print(f"Detected Lon columns: {lon_cols}")

    # 4. Attempt Audit if we can find the pairs
    # Adjust these strings if your columns have different names!
    try:
        # Assuming common naming patterns
        h_lat, h_lon = ('human_lat', 'human_lon') if 'human_lat' in df.columns else (lat_cols[0], lon_cols[0])
        o_lat, o_lon = ('oracle_lat', 'oracle_lon')
        
        df['error'] = simple_haversine(df[o_lat], df[o_lon], df[h_lat], df[h_lon])
        
        perfect_hits = df[df['error'] < 50]
        print(f"\n--- 📊 AUDIT RESULTS ---")
        print(f"Total Rows: {len(df)}")
        print(f"Bullseye Matches (<50m): {len(perfect_hits)}")
        
        if len(perfect_hits) > 0:
            sample = perfect_hits.sample(1).iloc[0]
            print(f"\n--- 🔍 MANUAL VERIFICATION ---")
            print(f"Instruction: {sample['instruction']}")
            print(f"👉 Link: https://www.google.com/maps/search/?api=1&query={sample[h_lat]},{sample[h_lon]}")
            
    except Exception as e:
        print(f"\n❌ Audit failed: {e}")
        print("Please check the column names printed above and update the script.")
else:
    print("❌ File not found at the specified path.")

--- 📂 FILE STRUCTURE ---
Columns found: ['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun', 'target_tags', 'city_source']
Detected Lat columns: []
Detected Lon columns: []

❌ Audit failed: list index out of range
Please check the column names printed above and update the script.


In [12]:
import pandas as pd
import networkx as nx
import pickle

# 1. Load the Parquet (The "What")
df = pd.read_parquet('../data/manhattan/manhattan_silver_standard_V4.parquet')

# 2. Load the Graph (The "Where")
# Using the gpickle file from your screenshot
with open('../data/manhattan/manhattan_graph.gpickle', 'rb') as f:
    G = pickle.load(f)

print(f"✅ Graph loaded with {len(G.nodes)} intersections.")

# 3. Mapping Function: Extract Lat/Lon from Graph Nodes
def get_node_coords(node_id):
    try:
        node = G.nodes[node_id]
        return node['y'], node['x'] # y=lat, x=lon in OSMnx
    except KeyError:
        return None, None

# 4. Hydrate the Parquet with physical coordinates
df['human_lat'], df['human_lon'] = zip(*df['gold_goal_node'].apply(get_node_coords))
df['start_lat'], df['start_lon'] = zip(*df['start_node'].apply(get_node_coords))

# 5. THE FINAL VERIFICATION
# This will pick a random instruction and show you exactly where it lands.
sample = df.dropna(subset=['human_lat']).sample(1).iloc[0]

print(f"\n--- 🗺️ SPATIAL VERIFICATION ---")
print(f"Instruction: {sample['instruction']}")
print(f"Target Node ID: {sample['gold_goal_node']}")
print(f"Actual Coordinates: {sample['human_lat']}, {sample['human_lon']}")
print(f"\n👉 VERIFY ON GOOGLE MAPS: https://www.google.com/maps?q={sample['human_lat']},{sample['human_lon']}")

C:\Users\adan\AppData\Local\Temp\ipykernel_6180\118848909.py:11: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_6180\118848909.py:11: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


✅ Graph loaded with 74137 intersections.

--- 🗺️ SPATIAL VERIFICATION ---
Instruction: from your location go stright and reach a park and the restaurant and opposite sideswingerclub then reache night club, just 5-6 blocks east of your current location. The theater is on the same block as the night club, meet me there.
Target Node ID: #6223203434
Actual Coordinates: 40.7207035, -73.9930241

👉 VERIFY ON GOOGLE MAPS: https://www.google.com/maps?q=40.7207035,-73.9930241


In [13]:
import numpy as np

# 1. THE HYDRATION: Map Graph IDs to Physical Coordinates
def get_node_coords(node_id):
    try:
        node = G.nodes[node_id]
        return node['y'], node['x'] 
    except:
        return None, None

print("🔄 Hydrating Parquet with spatial coordinates...")
df['human_lat'], df['human_lon'] = zip(*df['gold_goal_node'].apply(get_node_coords))
df['start_lat'], df['start_lon'] = zip(*df['start_node'].apply(get_node_coords))

# 2. THE HONEST ACCURACY: Self-contained Haversine for validation
def haversine_m(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]: return np.nan
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Calculate the actual trip distance and the current Oracle error
df['trip_distance_m'] = df.apply(lambda x: haversine_m(x.start_lat, x.start_lon, x.human_lat, x.human_lon), axis=1)

# 3. QUALITY TIERS: Categorize the data for your final report
df['quality_tier'] = 'Unverified'
df.loc[df['trip_distance_m'] < 500, 'quality_tier'] = 'Short Trip'
df.loc[df['trip_distance_m'] >= 500, 'quality_tier'] = 'Navigational Challenge'
df.loc[df['trip_distance_m'] > 2000, 'quality_tier'] = 'Long-Range Marathon'

# 4. FINAL STATS REPORT
print("\n--- 📈 HYDRATED DATASET PREVIEW ---")
print(f"Total Hydrated Rows: {len(df.dropna(subset=['human_lat']))}")
print(f"Median Trip Distance: {df['trip_distance_m'].median():.2f}m")
print("\nDistribution of Difficulty:")
print(df['quality_tier'].value_counts())

# Show one 'Navigational Challenge' to prove it's real
sample = df[df['quality_tier'] == 'Navigational Challenge'].sample(1).iloc[0]
print(f"\n✅ Verification Sample:")
print(f"Instruction: {sample['instruction'][:100]}...")
print(f"Trip Length: {sample['trip_distance_m']:.2f}m")

🔄 Hydrating Parquet with spatial coordinates...

--- 📈 HYDRATED DATASET PREVIEW ---
Total Hydrated Rows: 7000
Median Trip Distance: 1133.11m

Distribution of Difficulty:
quality_tier
Navigational Challenge    6283
Short Trip                 714
Long-Range Marathon          3
Name: count, dtype: int64

✅ Verification Sample:
Instruction: Meet me at the Rice N Beans Brazilian restaurant on 9th Avenue. There's a school just to the west of...
Trip Length: 987.21m


In [ ]:
# Save the 'Hydrated' version with the actual Lat/Lon columns
df.to_parquet('../data/manhattan/RVS_MANHATTAN_GOLD_HYDRATED.parquet')

print("🚀 STRATEGY COMPLETE.")
print("The file 'RVS_MANHATTAN_GOLD_HYDRATED.parquet' is now the source of truth.")
print("You can now evaluate any LLM against these 'human_lat' and 'human_lon' columns.")

🚀 STRATEGY COMPLETE.
The file 'RVS_MASTER_GOLD_HYDRATED.parquet' is now the source of truth.
You can now evaluate any LLM against these 'human_lat' and 'human_lon' columns.


In [16]:
# 1. Ensure the oracle columns exist. 
# If they are in a different dataframe (e.g., 'results_df'), merge them first:
# df = df.merge(results_df[['sample_id', 'oracle_lat', 'oracle_lon']], on='sample_id', how='left')

# If you just need to fill them for a test run, we can check if they exist:
if 'oracle_lat' not in df.columns:
    print("⚠️ 'oracle_lat' missing. Defaulting to 'start_lat' to test Anchor Bias...")
    df['oracle_lat'] = df['start_lat']
    df['oracle_lon'] = df['start_lon']

# 2. Optimized Error Calculation (Vectorized is faster and safer than .apply)
def calculate_metrics(dataframe):
    R = 6371000
    lat1, lon1 = np.radians(dataframe['oracle_lat']), np.radians(dataframe['oracle_lon'])
    lat2, lon2 = np.radians(dataframe['human_lat']), np.radians(dataframe['human_lon'])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    dataframe['dist_error_m'] = 2 * R * np.arcsin(np.sqrt(a))
    
    return dataframe

df = calculate_metrics(df)

# 3. Final Comparison
print(f"--- 📈 UPDATED CONVERGENCE AUDIT ---")
print(f"Bullseye Accuracy (<100m): { (df['dist_error_m'] <= 100).mean():.2%}")
print(f"Neighborhood Accuracy (<500m): { (df['dist_error_m'] <= 500).mean():.2%}")
print(f"Median Error: {df['dist_error_m'].median():.2f}m")

⚠️ 'oracle_lat' missing. Defaulting to 'start_lat' to test Anchor Bias...
--- 📈 UPDATED CONVERGENCE AUDIT ---
Bullseye Accuracy (<100m): 0.00%
Neighborhood Accuracy (<500m): 10.20%
Median Error: 1133.11m


In [17]:
# Check how many samples per city actually have coordinates calculated
coverage = df.groupby('city').agg({
    'human_lat': 'count', 
    'sample_id': 'count'
}).rename(columns={'human_lat': 'Hydrated', 'sample_id': 'Total'})

print("--- 🗺️ CITY HYDRATION COVERAGE ---")
print(coverage)

--- 🗺️ CITY HYDRATION COVERAGE ---
           Hydrated  Total
city                      
manhattan      7000   7000


In [20]:
import pandas as pd
import pickle
import os
import numpy as np

# 1. Update these to your actual paths!
city_graphs = {
    'manhattan': '../data/manhattan/manhattan_graph.gpickle',
    'pittsburgh': '../data/pittsburgh/pittsburgh_graph.gpickle',
    'philadelphia': '../data/philadelphia/philadelphia_graph.gpickle'
}

# 2. Load the Master Parquet
df = pd.read_parquet('../data/RVS_MASTER_SILVER_STANDARD.parquet')

# Initialize columns
for col in ['human_lat', 'human_lon', 'start_lat', 'start_lon']:
    df[col] = np.nan

# 3. The Hydration Loop
for city, graph_path in city_graphs.items():
    if not os.path.exists(graph_path):
        print(f"⚠️ Skipping {city}: File not found at {graph_path}")
        continue
    
    print(f"🏗️  Hydrating {city}...")
    with open(graph_path, 'rb') as f:
        G = pickle.load(f)
    
    def get_coords(node_id):
        try:
            node = G.nodes[node_id]
            # y=lat, x=lon is the standard for OSMnx gpickles
            return node['y'], node['x']
        except:
            return None, None

    mask = df['city'] == city
    # Map Goal Nodes
    goal_coords = df.loc[mask, 'gold_goal_node'].apply(get_coords)
    df.loc[mask, ['human_lat', 'human_lon']] = pd.DataFrame(goal_coords.tolist(), index=df.loc[mask].index).values
    
    # Map Start Nodes
    start_coords = df.loc[mask, 'start_node'].apply(get_coords)
    df.loc[mask, ['start_lat', 'start_lon']] = pd.DataFrame(start_coords.tolist(), index=df.loc[mask].index).values
    
    print(f"✅ {city} complete.")

# 4. Final Global Coverage Check
print("\n--- 🗺️ FINAL GLOBAL COVERAGE ---")
print(df.groupby('city')['human_lat'].count())

# 5. Save the truly complete file
df.to_parquet('../data/RVS_MASTER_GOLD_HYDRATED_FINAL.parquet')

🏗️  Hydrating manhattan...


C:\Users\adan\AppData\Local\Temp\ipykernel_6180\1854877623.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_6180\1854877623.py:28: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


✅ manhattan complete.
🏗️  Hydrating pittsburgh...


C:\Users\adan\AppData\Local\Temp\ipykernel_6180\1854877623.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_6180\1854877623.py:28: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


✅ pittsburgh complete.
🏗️  Hydrating philadelphia...


C:\Users\adan\AppData\Local\Temp\ipykernel_6180\1854877623.py:28: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


✅ philadelphia complete.

--- 🗺️ FINAL GLOBAL COVERAGE ---
city
manhattan       7000
philadelphia    1278
pittsburgh      1023
Name: human_lat, dtype: int64


In [22]:
import numpy as np

def calculate_global_metrics(dataframe):
    # 1. Handle missing oracle columns by testing the 'Anchor Bias' (Zero movement)
    if 'oracle_lat' not in dataframe.columns:
        print("⚠️ 'oracle_lat' missing. Using 'start_lat' as a baseline prediction.")
        dataframe['oracle_lat'] = dataframe['start_lat']
        dataframe['oracle_lon'] = dataframe['start_lon']
    
    # 2. Vectorized Haversine Math
    R = 6371000
    # Use .to_numpy() to ensure clean math even if there are index mismatches
    lat1, lon1 = np.radians(dataframe['oracle_lat'].values), np.radians(dataframe['oracle_lon'].values)
    lat2, lon2 = np.radians(dataframe['human_lat'].values), np.radians(dataframe['human_lon'].values)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    dataframe['dist_error_m'] = 2 * R * np.arcsin(np.sqrt(a))
    
    return dataframe

# Execute
df = calculate_global_metrics(df)

# 3. Final Global Report
print(f"\n--- 🌎 GLOBAL CONVERGENCE AUDIT (9,301 Samples) ---")
print(f"Global Bullseye (<100m): { (df['dist_error_m'] <= 100).mean():.2%}")
print(f"Global Neighborhood (<500m): { (df['dist_error_m'] <= 500).mean():.2%}")
print(f"Global Median Error: {df['dist_error_m'].median():.2f}m")

print("\n--- 🏙️ PERFORMANCE BY CITY ---")
city_stats = df.groupby('city')['dist_error_m'].agg(['count', 'median', 'mean'])
print(city_stats)

⚠️ 'oracle_lat' missing. Using 'start_lat' as a baseline prediction.

--- 🌎 GLOBAL CONVERGENCE AUDIT (9,301 Samples) ---
Global Bullseye (<100m): 0.00%
Global Neighborhood (<500m): 11.13%
Global Median Error: 1117.41m

--- 🏙️ PERFORMANCE BY CITY ---
              count       median         mean
city                                         
manhattan      7000  1133.108882  1098.936976
philadelphia   1278  1135.927787  1096.661052
pittsburgh     1023   954.097607   960.517929


In [ ]:
# Save the final, unified Master Gold Dataset with my preferred name
df.to_parquet('../data/RVS_MASTER_GOLD_HYDRATED.parquet')

print("🏅 PROJECT MILESTONE REACHED 🏅")
print("File saved as: RVS_MASTER_GOLD_HYDRATED.parquet")
print(f"Final Row Count: {len(df)}")
print(f"Cities included: {df['city'].unique().tolist()}")

🏅 PROJECT MILESTONE REACHED 🏅
File saved as: RVS_MASTER_GOLD_HYDRATED.parquet
Final Row Count: 9301
Cities included: ['philadelphia', 'pittsburgh', 'manhattan']
